# Lesson 3: Standalone Session from DataFrames

**Module:** AGA Fundamentals  
**Dataset:** Movies (synthetic — built from Pandas DataFrames in this notebook)

A *standalone* session has no AuraDB attached. You feed it Pandas DataFrames, run algorithms, and pull results back to Python. Useful when your data isn't in Neo4j yet — a CSV export, a Parquet file, anything you can land in Pandas.

We'll build a small Actor-COLLABORATED-Actor graph from two DataFrames and run PageRank on it.

## Setup

Recreate the imports, credentials, and `sessions` manager from the previous lessons.

Note: standalone sessions don't read from AuraDB, so `AURA_URI`, `AURA_USERNAME`, and `AURA_PASSWORD` aren't strictly required here — they're loaded for consistency with the other notebooks.


In [ ]:
# Recreate the notebook setup from the previous lessons:
#   1. Import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, SessionMemory
#      from graphdatascience.session.
#   2. Import load_dotenv from dotenv, plus os, timedelta, pandas, display.
#   3. Call load_dotenv() and read your Aura API + AuraDB credentials into
#      local variables (client_id, client_secret, uri, username, password).


In [ ]:
# Build the sessions manager with your Aura API credentials — see Lessons 1 and 2.


## Create the standalone session

To construct a projection from DataFrames, you'll need to create a self-hosted session. This session is just like any other, except it is not connected to any AuraDB.

To create a self-hosted session, you'll:

* Remove the `db_connection` parameter
* Add a `cloud_location` parameter

`CloudLocation` is imported above and used in the cell below.

In [ ]:
gds = sessions.get_or_create(
    session_name="standalone-demo",
    memory=SessionMemory.m_2GB,
    ttl=timedelta(minutes=30),
    cloud_location=CloudLocation("gcp", "europe-west1"),
)
gds.verify_connectivity()
print(f"Connected to GDS Session: {gds}")

## Where your session lives

If you check your **Aura Graph Analytics workspace** in the Aura Console, you should now see a session appear — labelled as **self-hosted**. It's marked "self-hosted" because it's not tied to any AuraDB.

## Build the DataFrames

To project into a self-hosted session, you'll use `gds.graph.construct(...)`. It expects two specific column conventions:

* **Nodes frame**: `nodeId` (int) and `labels` (str).
* **Relationships frame**: `sourceNodeId` (int), `targetNodeId` (int), `relationshipType` (str).

In either frame, you are free to add extra numeric properties. This way, you could construct an identical graph to anything you might project from a Neo4j database.

String properties cannot be projected. Drop columns like `name` or `title` before passing — you can merge them back into the result after streaming.

In [ ]:
actors = pd.DataFrame({
    "nodeId": [0, 1, 2, 3, 4, 5],
    "labels": ["Actor"] * 6,
    "born":   [1953, 1962, 1956, 1964, 1958, 1971],
})

actor_names = pd.DataFrame({
    "nodeId": [0, 1, 2, 3, 4, 5],
    "name":   ["Tom Hanks", "Tom Cruise", "Meryl Streep",
               "Sandra Bullock", "Kevin Bacon", "Winona Ryder"],
})

edges = pd.DataFrame({
    "sourceNodeId":     [0, 0, 1, 2, 2, 3, 4],
    "targetNodeId":     [4, 2, 3, 4, 5, 5, 5],
    "relationshipType": ["COLLABORATED"] * 7,
    "collaborations":   [3.0, 1.0, 2.0, 1.0, 4.0, 2.0, 1.0],
})

print(f"Actors: {len(actors)} | Edges: {len(edges)}")

## Construct the projection

With the frames ready, the projection is a single call: `gds.graph.construct(...)`.

In [ ]:
G = gds.graph.construct("collaborations", actors, edges)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

## See your graph

A six-node toy graph is easy to picture, but most real projections are not. The `neo4j-viz` package renders your projection as an interactive force-directed visualization right in the cell output.

In [ ]:
from neo4j_viz.gds import from_gds

VG = from_gds(gds, G)
VG.render()

## Stream is your only output route

In standalone mode, `stream` is the only mode that gets results out of the session. There's no database for `write` to write to.

In [ ]:
df = gds.pageRank.stream(
    G,
    relationshipWeightProperty="collaborations",
)
display(df.sort_values("score", ascending=False))

The DataFrame has `nodeId` and `score` columns. `nodeId` is the integer you assigned in the actors frame.

## Merge names back from the side frame

The result has raw `nodeId`s. Run the following cell to merge them back to `actor_names`.

In [ ]:
ranked = (
    df.merge(actor_names, on="nodeId", how="left")
      .sort_values("score", ascending=False)
      [["name", "score"]]
)
display(ranked)

You can now use the resulting DataFrame as you would any other.

## Clean up

Standalone sessions terminate the same way as attached ones. Nothing in a standalone session persists once it ends — capture anything you need to keep from `stream` results before calling `delete()`.

In [ ]:
sessions.delete(session_name="standalone-demo")
print("Session deleted.")

That's it — the full standalone workflow.

You've completed Module 3:

* Authenticated against the Aura API from Python (`1_connect`).
* Run the full attached AGA workflow against AuraDB (`2_syntax`).
* Run an algorithm on data that isn't in Neo4j yet (`3_standalone`).

You're now equipped to script any AGA workflow end to end.